In [1]:
import pandas as pd
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
## Definition of a hyperparameter grid:
learning_rate_variants = [0.1, 0.01]
batch = [10, 20, 30]
hidden_layer_variants = [[100, 100], [100, 100, 100]] #Füllwerte aktuell
input_layer = 10
epochs = 10

In [3]:
## Import the data-splits:
df_train = pd.read_parquet("../data/df_train.parquet")
df_val = pd.read_parquet("../data/df_val.parquet")
df_test = pd.read_parquet("../data/df_test.parquet")
print(f"Train split loaded. Shape: {df_train.shape}")
print(f"Validation split loaded. Shape: {df_val.shape}")
print(f"Test split loaded. Shape: {df_test.shape}")

Train split loaded. Shape: (4641696, 36)
Validation split loaded. Shape: (1852320, 36)
Test split loaded. Shape: (2778934, 36)


In [4]:
# DEFINE TARGET AND FEATURES
target = 'Total_Trip_Starts'  # Assuming this is the target variable in the dataset

# Select final features and target column
features = [
    'distance_to_loop',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'is_weekend', 'is_holiday', 'day_of_week',
    'poi_cat_automotive', 'poi_cat_civic_community', 'poi_cat_education',
    'poi_cat_entertainment', 'poi_cat_finance', 'poi_cat_food_drink',
    'poi_cat_grocery', 'poi_cat_health', 'poi_cat_leisure_sports',
    'poi_cat_lodging', 'poi_cat_nightlife', 'poi_cat_services','poi_cat_shopping', 'poi_cat_transport'
]

# Todo: dataset preparation

# train_set =
# validation_set =
# test_set =

# train_loader = torch.utils.data.DataLoader(train_set, batch_size = batch)
# test_loader = torch.utils.data.DataLoader(test_set, batch_size = batch)
# validation_loader = torch.utils.data.DataLoader(validation_set, batch_size = batch)

# Scaling
scaler = StandardScaler()
scaler.fit(df_train[features])

X_train = scaler.transform(df_train[features])
y_train = df_train['Total_Trip_Start'].values

X_val = scaler.transform(df_val[features])
y_val = df_val['Total_Trip_Start'].values

X_test = scaler.transform(df_test[features])
y_test = df_test['Total_Trip_Start'].values



In [5]:
# Definition of the feedforward NN:
# Notes: We make the number of hidden layers with the number of neurons dynamic, so we can try out different complexities in our grid-search
class FNN(nn.Module):
    def __init__(self, input_size, hidden_layers):
        super(FNN, self).__init__()
        hidden_layers.insert(0, input_size)
        zip_layers = zip(hidden_layers[:-1], hidden_layers[1:])
        self.layers = nn.ModuleList([
            nn.Linear(in_neurons, out_neurons) for in_neurons, out_neurons in zip_layers
        ])
        self.output_layer = nn.Linear(hidden_layers[-1], 1)

    def forward(self, x):
        for layers in self.layers:
            x = F.relu(layers(x))
        return self.output_layer(x)


In [6]:
# Definition of the training loop:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")

def train_model(FNN, device, train_loader, optimizer, loss_funct, epochs):
    FNN.train()
    during_training_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = FNN(inputs)
        loss = loss_funct(outputs, labels)
        loss.backward()
        optimizer.step()
        during_training_loss += loss.item()
        print(f'TRAINING Epoch {epochs}, Batch {i+1}, Loss: {during_training_loss}')
        during_training_loss = 0.0


In [7]:
# TODO: write validation loop

def validate_model(FNN, device, validation_loader, loss_funct):
    FNN.train()
    validation_loss = 0.0
    for inputs, labels in validation_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = FNN(inputs)
        loss = loss_funct(outputs, labels)
        validation_loss += loss.item()
    print(f'VALIDATION, Loss: {validation_loss}')

In [ ]:
# Grid-search for each hyperparameter combination defined before.
# Train the model for each batch in one epoch and validate the model after each epoch on the validation set

loss_funct = nn.MSELoss()
for hidden_variant in hidden_layer_variants:
    for learning_rate in learning_rate_variants:
        model = FNN(input_layer, hidden_variant)
        optimizer = optim.SGD(model.parameters(), lr=learning_rate)
        for epoch_index in range(1, epochs+1):
            train_model(model, device, train_loader, optimizer, loss_funct, epoch_index)
            validate_model(model, device, validation_loader, loss_funct)
